In [1]:
import random
import numpy as np

# Envirenment

In [2]:
NUM_STATES = 5                           # Number of states
NUM_ACTIONS = 2

ALPHA = 0.1
GAMMA = 0.9
EPSILON = 0.1

START_STATE = 0
GOAL_STATE = 4
TERMINAL_STATE = 4                          

LEFT = 0
RIGHT = 1

In [3]:
def step(state, action, goal_state=GOAL_STATE):
    """
    Applies an action and returns: next_state, reward, terminated
    """
    if action == 0:
        state -= 1
        if state < 0 : state = 0
    elif action == 1:
        state += 1
        if state > goal_state: state = goal_state
    
    if state == goal_state:
        reward = 1
    else:
        reward = 0
    
    if state == goal_state: terminated = True
    else: terminated = False
    
    return state, reward, terminated

# Policy

In [4]:
def greedy_action(state, Q):
    """
    Return the action with the larger estimated Q-value for specified state.
    """
    if Q[state, LEFT] < Q[state, RIGHT]:
        return RIGHT
    elif Q[state, LEFT] > Q[state, RIGHT]:
        return LEFT
    else:
        return int(random.randint(0, 1))

In [5]:
def choose_action(state, Q, epsilon=EPSILON):
    """
    Choose an action using epsilon-greedy exploration.
    """
    if random.random() < epsilon:
        return random.randint(0, 1)
    else:
        return greedy_action(state, Q)

# SARSA

In [6]:
def sarsa_update(
        Q, state, action, reward, 
        next_action, next_state, done, 
        alpha=ALPHA, gamma=GAMMA,
):
    """
    Perfoms one SARSA update.
    """

    if done:
        target = reward
    else:
        next_value = Q[next_state, next_action]
        target = reward + gamma * next_value
    
    td_error = target - Q[state, action]
    Q[state, action] += alpha * (td_error)

    return Q, td_error

# Episode Generation

In [7]:
def run_sarsa_episode(Q, epsilon=EPSILON, alpha=ALPHA, gamma=GAMMA , max_steps=100):
    """
    Runs one episode and return its total rewards, number of steps until termination, Q matrix, and td error for every step.
    """    
    state = START_STATE
    action = choose_action(state, Q, epsilon)
    total_reward = 0
    td_errors = []

    for step_idx in range(max_steps):

        next_state, reward, terminated = step(state, action, GOAL_STATE)
        total_reward += reward

        if terminated:
            next_action = action
            Q, td_error = sarsa_update(Q, state, action, reward, next_action, next_state, terminated, alpha, gamma)
            td_errors.append(td_error)
            return total_reward, step_idx + 1, Q, td_errors 
        
        next_action = choose_action(next_state, Q, epsilon)
        Q, td_error = sarsa_update(Q, state, action, reward, next_action, next_state, terminated, alpha, gamma)
        td_errors.append(td_error)
        state = next_state
        action = next_action

    return total_reward, max_steps, Q, td_errors 

# Training

In [8]:
Q = np.zeros((NUM_STATES, NUM_ACTIONS), dtype=float)
num_episodes = 5000
episode_length = []
rewards = []
errors = []
for episode in range(num_episodes):
    reward, length, Q, td_error = run_sarsa_episode(Q, EPSILON, ALPHA, GAMMA)
    episode_length.append(length)
    rewards.append(reward)
    errors.append(td_error)


# Results

In [9]:
print('Q:')
print(Q)

Q:
[[0.63025045 0.69844231]
 [0.61862321 0.80625086]
 [0.70269838 0.89951463]
 [0.77130627 1.        ]
 [0.         0.        ]]


In [10]:
print('episode_length:')
print(episode_length)

episode_length:
[11, 10, 5, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 4, 4, 4, 4, 4, 4, 4, 6, 4, 4, 4, 4, 4, 4, 4, 6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 4, 4, 4, 4, 4, 10, 6, 4, 4, 4, 4, 6, 5, 4, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 4, 4, 4, 6, 4, 4, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 4, 4, 4, 5, 4, 4, 4, 4, 4, 6, 4, 4, 4, 6, 4, 6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8, 4, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 5, 6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 4, 4, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 4, 4, 4, 4, 4, 6, 4, 4, 4, 4, 4, 4, 4, 4, 6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 4, 4, 4, 4, 6, 6, 4, 4, 4, 6, 4, 4, 6, 4, 4, 4, 4, 4, 4, 4,

In [11]:
print('rewards:')
print(rewards)

rewards:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [12]:
print('td errors:')
print(errors)

td errors:
[[np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.0)], [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.09000000000000001), np.float64(0.9)], [np.float64(0.0), np.float64(0.0), np.float64(0.008100000000000001), np.float64(0.162), np.float64(0.81)], [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0007290000000000002), np.float64(0.02187), np.float64(0.21870000000000003), np.float64(0.729)], [np.float64(0.0026244000000000007), np.float64(0.039366000000000005), np.float64(0.26244000000000006), np.float64(0.6560999999999999)], [np.float64(0.005904900000000002), np.float64(0.05904900000000002), np.float64(0.295245), np.float64(0.59049)], [np.float64(0.010628820000000002), np.float64(0.07971615000000001), np.float64(0.31886460000000

# Tests

In [13]:
# Environment
assert step(0, RIGHT) == (1, 0, False)
assert step(3, RIGHT) == (4, 1, True)

# Learned ordering
for state in range(4):
    assert Q[state, RIGHT] > Q[state, LEFT]

# Goal transition
assert np.isclose(Q[3, RIGHT], 1.0)

In [14]:
for state in range(4):
    print(state, Q[state], greedy_action(state, Q))

0 [0.63025045 0.69844231] 1
1 [0.61862321 0.80625086] 1
2 [0.70269838 0.89951463] 1
3 [0.77130627 1.        ] 1


# Evaluation

In [15]:
def evaluation(Q, num_episodes=100, max_steps=100):
   
    episode_length = []
    episode_rewards = []
  
    for episode in range(num_episodes):

        state = START_STATE
        length = max_steps    
        total_reward = 0                                             

        for step_idx in range(max_steps):
            action = greedy_action(state, Q)
            next_state, reward, terminated = step(state, action, GOAL_STATE)
            
            total_reward += reward
            state = next_state
            if terminated:
                length = step_idx + 1
                break  
    
        episode_length.append(length)
        episode_rewards.append(total_reward)
    return episode_length, episode_rewards

In [16]:
Q_before = Q.copy()
eval_episode_length, eval_total_reward = evaluation(Q)
print('evaluation episode_length:')
print(eval_episode_length)
print('evaluation total rewards:')
print(eval_total_reward)
print("Unique evaluation lengths:", np.unique(eval_episode_length))
print("Mean evaluation length:", np.mean(eval_episode_length))

evaluation episode_length:
[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
evaluation total rewards:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Unique evaluation lengths: [4]
Mean evaluation length: 4.0


In [17]:
assert np.allclose(Q, Q_before)